In [3]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from catboost import CatBoostClassifier
from sklearn.experimental import enable_hist_gradient_boosting 
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score
import shap


In [ ]:
# Load the Iris dataset
df = pd.read_csv('selected_df.csv')
#encode data:
# label_encoder = LabelEncoder()
# df['ecoli_risk'] = label_encoder.fit_transform(df['ecoli_risk'])
#and any others

X = df.drop(labels = 'ecoli_risk', axis = 1)
y = df.ecoli_risk


# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the continuous data features using StandardScaler if we need it
# scaler = StandardScaler() #optiona;
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Define the hyperparameter search space, random or grid search or nested search
param_dist = {
    'C': np.logspace(-4, 4, 100),  
    'penalty': ['l2'],            
    'solver': ['lbfgs']           
}

# Create the mode
model = LogisticRegression(max_iter=1000)


# Define inner and outer cross-validation strategies
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)  # Inner loop for hyperparameter tuning
outer_cv = KFold(n_splits=3, shuffle=True, random_state=42)  # Outer loop for performance estimation

# Create a GridSearchCV object for the inner loop
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=inner_cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
# OR Perform RandomizedSearchCV with error_score='raise' 
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    scoring='accuracy',
    random_state=21,
    error_score='raise'
)

#---------------------CV-----------------------
# Perform nested cross-validation using cross_val_score
nested_scores = cross_val_score(
    estimator=grid_search,
    X=X_train,
    y=y_train,
    cv=outer_cv,
    scoring='accuracy',
    n_jobs=-1
)

# Output the nested cross-validation results
print(f'Nested CV Accuracy: {nested_scores.mean():.3f} (+/- {nested_scores.std() * 2:.3f})')

#--------------------BEST MODEL----------------------
# After nested CV, refit the best model on the entire training set
# The best estimator from the inner loop is automatically refit on the full training data
best_model = grid_search.fit(X_train, y_train)

# Now use the best model to make predictions on the test set
test_predictions = best_model.predict(X_test)
test_accuracy = (test_predictions == y_test).mean()
print(f'Test Accuracy: {test_accuracy:.3f}')


# Fit the RandomizedSearchCV
#random_search.fit(X_train, y_train)


In [ ]:
# Perform 5-fold cross-validation using accuracy as the scoring metric
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')

# Print the accuracy scores for each fold
print("Accuracy scores by fold:", cv_scores)

# Print the mean accuracy and standard deviation
print("Mean Accuracy: {:.4f} (+/- {:.4f})".format(cv_scores.mean(), cv_scores.std() * 2))

# XAI

In [ ]:
#shap stuff